# Lecture about R
In this lecture we will try the linear probability models, and do some tests that might be useful to you with your project.

### Clean your environment
This first block is mostly **necessary if you are running R locally in your comuter***(for example using RStudio)*

In [ ]:
rm(list=ls(all=TRUE))
graphics.off()
close.screen(all = TRUE)
erase.screen()
windows.options(record=TRUE)

### Set your working directory
This is another thing that is **relevant *iff* you are working locally**. You need to let R know where are your files. If you do not this, you will have to provide absolute paths to your files at every time. You could go around this creating a variable with your PATH, but setting the working directory will prove useful to save clutter.

In [ ]:
setwd("/YOUR/PATH/TO/THE/FILES")

### Import the data
1. Now we arrive to the core of the issue. We need to import the data. First, we need to have the data available to us. See that folder icon on the "left" bar? Click on that, and drop your files (if you prefer, you can create a folder and put them there).
2. Now that you have the data, we will need the "PATH" to it, and for that, right click on the file and select **copy path**. This will put in the clipboard the PATH to the file.
3. Load a library that will allow you to read an excel file (as our file is an xlsx file). Some libraries are already intalled in Google's Colab, but many are not. First try to load the library, if it does not work, you can use the command `install.packages('name_of_the_library')` to install it, we will do this later.

In [ ]:
library(readxl)

#### Store the data in a "variable"
Well, here I am abusing the work "variable" a little bit, but the idea is that we will put a name to our data. Here, it is going to be `apple`. For that, we write the name for our data, a left pointing arrow with a lower than and an ephen `<-` and then the command `read_excel(path_to_file, sheet=name_of_sheet)`. Remember how we got the PATH to the file above? Well it is a good opportunity to paste it here.

In [ ]:
path_to_apple <- '/kaggle/input/datasets/pifr86/apple-dataset/apple.xlsx'
path_to_apple_details <- '/kaggle/input/datasets/pifr86/apple-dataset/apple_details.xlsx'

apple <- read_excel(path_to_apple, sheet = "Sheet1")
apple_details <- read_excel(path_to_apple_details, sheet = "Sheet1")

### Analyze the data
To analyze our data, R has a very nice package that will allow us to present summary statistics and regression tables in a very easy to read format. Not only that, it will make exporting this data into a format that we can import into a document a breeze. This package is called `stargazer`. Let's try to load it.

In [ ]:
library(stargazer)

If you did this locally in your computer or in Google Colab, it did not work, because `stargazer` is one of the packages that is not installed by default in R or Google's Colab [luckily it is in Kaggle]. To solve this let's install it using the command `install.packages()` that we saw earlier. Here in Kaggle, most *relevant* packages are preinstalled.

In [ ]:
install.packages('stargazer')
library(stargazer)

Good, now we can use stargazer. To start, we need to ackowledge that the fundamental way in which R handles data is by organizing it in **dataframes**.

In this case, while `apple` is conceptually a dataframe after using `read_excel()`, sometimes the object returned by specific import functions like `read_excel()` might have additional attributes or a slightly different internal structure that stargazer isn't designed to handle directly. Explicitly using `as.data.frame()` ensures that stargazer receives a standard R dataframe object, guaranteeing compatibility and preventing potential errors. It's a good practice to explicitly convert when using functions that expect a standard dataframe. This is why we are including here `as.data.frame()` around `apple`.

We could have done this separately, as something like `apple <- as.data.frame(apple)` and from there apple would be already correctly formatted as a dataframe, and we would not need to include that part in this command.

Anyway, then we need to tell stargazer how we can to see the information (by default it will create a LaTeX table, which is out of the scope of this course, although it should not be 🤓). If we want just to see the output, use the argument `type="text"`. Next you can specify the number of digits, in this case 2, and if you want to include the median or not, using the boolean `TRUE` or `FALSE` (this is also optional).

In [ ]:
stargazer(as.data.frame(apple), type = "text",
          digits=2, median = TRUE)

What about if you want to store this data to reuse it later in your final document? Well in that case we need to ask Stargazer to create a file that we can reuse. Now change `type="html"` to create an HTML document, and add an extra term called `out="name_of_your_file.html"`. In this case we decided to name the file `Stats.html`. This new file will be store alongside the excel files you uploaded recently. Loon on the left after you run the command.

In [ ]:
stargazer(as.data.frame(apple), type = "html",
          digits=2, median = TRUE,
          out="Stats.html")

What you see in the output window is the contents of the file. In this case the same table that was above, but formatted with HTML. You could copy and paste that code in a new file, with extension HTML and then open it with your browser and you would obtain the same think you get from the file that was just created.

### Distribution of the data
Another very important piece of information regarding our data, is how it is distributed. To achieve that, we could use some histograms. Getting a histogram with R is quite simple, we just use the command `hist(your_data)`. Here we will also take the opportunity to see how we can work with a dataframe. We know that `apple` is a dataframe. This data, has several variables, for example `regprc` or `ecoprc`. How can we access them? Well R uses the dollar sign for this. We write first the name of the dataframe, the dollar sign, and the name of the variable. This will return us only the variable we are interested in. Let's see the example below creating histograms for `regprc` and `ecoprc`

In [ ]:
hist(apple$regprc)
hist(apple$ecoprc)

Plotting some figures can help you with:
1. Identification of outliers
2. Skew of the distribution (should you use logs?)
3. Understand if the relationship between $x$ and $y$ is non-linear (should I include a $x^2$? or use logs?)

In [ ]:
plot(apple$ecoprc, apple$regprc)

Of course in this case we have a scatter plot that looks like this because our dependant variable is discrete!

Maybe we want to overview our data, to see how does it looks, but without opening the whole dataset. For that R has a function named `head()`. This will list the first 6 rows of our dataframe. If you used python's pandas before you will recognize this function, actually pandas is a clone for python of a very old version of R.

In [ ]:
head(apple)

SUppose here we would like to represent the `faminc` (family income variable) in tenths. For that we need to replace it by itself divided by 10. This will look weird mathematically $$faminc=\frac{faminc}{10}$$ You see this is not an equation, but it will read like *new faminc will be old faminc divided by 10*. If we want to store this in `faminc` we need to use the assignment process we hade before, the left arrow `apple$faminc <- apple$faminc/10`

In [ ]:
apple$faminc <- apple$faminc/10
head(apple)

We could be interested in having a variable that indicates if the customer bought some eco products. This will be the case if ecolbs (the weight of the eco products bought) is larger than zero. R has the function `ifelse(condition, iftrue, iffalse)`. In this case, our condition will be `apple$ecolbs>0`, we want a `1` if this is true, and a `0` if this is false. As you know 1 and 0 are the logical equivalents to `TRUE` and `FALSE`. Again, we will store this result in a variable named ecobuy.

In [ ]:
apple$ecobuy <- ifelse(apple$ecolbs > 0, 1, 0)
head(apple)

### Partition our data
Maybe we do not want to consider all the variables in our dataframe. We can use the function `subset(data, select = c(variables_separated_by_commas))` like below. We will store a subset of our data into `mydata`. This will be a new dataframe such that $mydata\subset apple$. Let's immediately check the summary statistics with this reduced set of data.

In [ ]:
mydata <- subset(apple, select = c(ecobuy,ecoprc,regprc,faminc,hhsize,educ,age))

stargazer(as.data.frame(mydata), type = "text",
          digits=2, median = TRUE)

Let's store this, as we know how to do.

In [ ]:
stargazer(as.data.frame(mydata), type = "html",
          digits=2, median = TRUE,
          out="Stats2.html")

### Correlation matrix
A very important information we would like to have about our data, is potential (high) correlations between our explanatory variables. For that, we will produce a correlation matrix using the function `corr(data)`. To see the output we just reproduce the name of the variable we used to store the output of that function.

In [ ]:
mydata.cor = cor(mydata)
mydata.cor

We can use stargazer to store this information as well.

In [ ]:
stargazer(mydata.cor, title="Correlation Matrix",
          digits = 2,
          type = "html", out ="corr.html")

We can have this information in a more graphical way, using the library `corrplot`. Let's install it (it is not by default available in Colab) and load it.

In [ ]:
install.packages('corrplot')
library(corrplot)

We will use it like this: `corrplot()` and in the arguments, we start with a correlation matrix data, in this case `mydata.cor`. The second argument `method="color"` indicates that we want to use color to signal higher and lower correlations. `type="upper"` indicates that we want an upper triangular matrix (change it to lower and see what happens! remove the command and se what happens!). `tl.col="black"` is there to make the name variables black, by default they would be red, not very reader friendly. Finally `tl.srt=45` makes the top row to be diagonally oriented (45 degrees), purely to make this more aesthetically pleasant.

In [ ]:
corrplot(mydata.cor, method = "color", type = "upper", tl.col = "black", tl.srt = 45)

There are two ways to store this figure. Right click and save the image, or you can use the `png(name_of_file.png, width=desired_width, height=desired_height)` function from R like this, you open a png graphics device, plot the correlation plot with the command above, and then close with dev.off().

In [ ]:
png("correlation_plot.png", width = 800, height = 600)
corrplot(mydata.cor, method = "color", type = "upper", tl.col = "black", tl.srt = 45)
dev.off() # Close the graphics device

## Regression!
Ok, enough of studying the data and let's run some regressions. The command for R to run a linear regression is the linear model `lm(y~x1+x2+..., data= data)` function. In this case our $y$ is `ecobuy`, if the customer bought or not an eco product. We are regressing this on `ecoprc`, `regprc`, `faminc`, `hhsize`, `educ`, and `age`. Of course, our data is stored in the dataframe `mydata`. Our model will be the following:
$$ecobuy=\beta_0+\beta_1 ecoprc+\beta_2 regprc+ \beta_3 faminc+\beta_4 hhsize+\beta_5 educ+\beta_6 age+u$$ in order to see the output we will use `summary()` again. We will store our linear model into the variable `ols`.

In [ ]:
ols <- lm(ecobuy~ecoprc+regprc+faminc+hhsize+educ+age, data = mydata)
summary(ols)

We can again use stargazer to store the results of our regression:

In [ ]:
stargazer(ols, type = "html", out ="ols.html")

### F-test of joint significance tests
We might be interested to check if ecoprc and regprc are both 0 simoultaneously. For that we know we need to run an $F$ test. We will need the `car` library, that we also need to install as Google Colab does not have it by default.

In [ ]:
install.packages('car')
library("car")

We now can run some joint significance tests! Let's test if $\beta_1=\beta_2=0$. To do that, we need to use the function `linearHypothesis(model, c(hypotheses))`. In R `c()` is a vector, and we need to use strings to include our restrictions, in this case we will say that `"ecoprc=0"` and `"regprc=0"`. Note that we are refering to the betas of these variables in the regression.

In [ ]:
linearHypothesis(ols, c("ecoprc=0","regprc=0"))

We see clearly that we can reject our hypothesis.

We can also test if one effect is the symmetric of the other, and this works very literally! in this case we would need to use the restriction `"ecoprc=-regprc"`

In [ ]:
linearHypothesis(ols, c("ecoprc=-regprc"))

We cannot reject the null hypothesis, or the fact that one is the negative of the other.

## Getting $\hat{y}$ and more stuff from a linear regression
When we ran the linear model and stored it in `ols`, what we did was to store a bunch of stuff into it. In particular we stored the fitted values and the residuals. Well, these fitteed values are $\hat{y}$. Using `names()` we can see all we can obtain from our linear model.

In [ ]:
names(ols)

Now let's save the $\hat{y}$

In [ ]:
mydata$yhat.ols <- ols$fitted.values

If we want to see the interval for our $\hat{y}$ we use the function `range()`

In [ ]:
range(mydata$yhat.ols)

What is the problem with these values?

In [ ]:
sum(mydata$yhat.ols>1)

## Homoskedasticity
We will have to deal with homoskedasticity in our model, be it testing for it, or fixing our standard errors. We will be using some libraries to do so: `lmtest` and `skedastic`.

`lmtest` will provide us with a function to do a Breusch-Pagan test, named `bptest(model)` where *model* stands for the name of our regression.

In [ ]:
install.packages('lmtest')
library(lmtest)

In [ ]:
bptest(ols)

We see that we can reject the null, it seems we might have heteroskedasticity! Let's try now the white test. For that we use `skedastic`'s function `white(model, interactions = BOOLEAN)` where we can include or exclude the interactions. In this case we will include them, using `TRUE` in the function.

In [ ]:
install.packages('skedastic')
library(skedastic)

In [ ]:
white(ols, interactions = TRUE)

Remember to be careful with the degrees of freedom as you lose a lot with this test!

### The Lazy White test.
This test unfortunately we need to do it by hand. On the good side, it is fairly yeasy to do so. First, we need to use our $\hat{y}$, square them, and then regress the squared residuals $\hat{u}^2$ on $\hat{y}$ and $\hat{y}^2$.

In [ ]:
mydata$yhat.ols_sq<-(ols$fitted.values)^2
mydata$ols.residualssq <- (ols$residuals)^2

Now we can regress $$\hat{u}^2=\alpha_0+\alpha_1 \hat{y}+\alpha_2\hat{y}^2+\epsilon$$ let's call this linear model `lazy_white` and check the output with `summary()`

In [ ]:
lazy_white<-lm(ols.residualssq~yhat.ols+yhat.ols_sq, data=mydata)
summary(lazy_white)

Now we need to check the significance. Remember, we want to do an $F$ test, for that we can use the $n\times R^2$ test, that is distributed $\chi^2_2$. To get the $R^2$ we can reuse the data that is in the summary of our regression. Let's check.

In [ ]:
names(summary(lazy_white))

Now we can use the `r.squared` and `nobs` to see our statistic $nR^2$.

In [ ]:
s <- summary(lazy_white)$r.squared*nobs(lazy_white)
s

To find the critical value, we can use the `qchisq(quartile, dof)` function. In this case we could use `qchisq(0.95,2)`.

In [ ]:
qchisq(0.95,2)

Alternatively we could use the function `pchisq()` where we give a value for $x$, the $dof$ and we obtain the probability associated to that distribution. We stressed further here that we do not want to have the lower tail, although this is unnecessary, as the default is the cumulative function.

In [ ]:
pchisq(q=s, df=2, lower.tail=FALSE)

So we cannot reject the null, and have heteroskedasticity.

## Fixing the problem
### Weighted least squares
We can try to fix the issue here using WLS. First, lets fix the situations where we had out of range predicted data. Also preventing dividing by 0.

In [ ]:
mydata$fit <- mydata$yhat.ols

In [ ]:
mydata$fit[mydata$fit<=0] <- 0.01
mydata$fit[mydata$fit>=1] <- 0.99

In [ ]:
range(mydata$fit)

Now we need to compute the weights, for that we need to compute the variance of the predicted variable $p(1-p)$ so we can devide our regression on both sides by it. Note that $V[y|x]=V[u|x]$ because $X\beta$ is deterministic, and then, conditional on $X$, it does not have a variance.

In [ ]:
mydata$hi<-((mydata$fit)*(1-mydata$fit))
mydata$weight_h <- 1/(mydata$hi)

To run a wls, we use the same function we used for ols, the difference is that we add the argument `weight=weight_h` indicating which weights we want to use in our regression.

In [ ]:
wls = lm(ecobuy~ecoprc+regprc+faminc+hhsize+educ+age, weight=weight_h, data = mydata)
summary(wls)

We can trust in these standard errors and make inference on this.

### Robust standard errors

We can compute robust standard errors

In [ ]:
install.packages('sandwich')
library(sandwich)

The command `vcocvHC()` generates, from a regression, `ols` in this case, the matrix $$\frac{n}{n-k} (X'X)^{-1}X'\hat{\Omega}X(X'X)^{-1}$$ $HC1$ indicates the multiplication at the begining, so we adjust for small samples and avoid a biased estimator (like when dividing by $n-1$ for the sample variance). In this case $\hat{\Omega}=diag(\hat{u}_1^2,...\hat{u}^2_n)$

In [ ]:
vcHC <- vcovHC(ols, type = "HC1")

Now we can obtain the robust standard errors by taking the square root (from *variance* to *standard deviation*)

In [ ]:
robust_se <- sqrt(diag(vcHC))

Finally, let's compare the models. Note that we are giving now 3: $ols$ twice, and $wls$. Note that in the argumnet `se=list(NULL,robust_se,NULL)` we are telling to use, in the second ols model, the robust standard errors we just computed above!

In [ ]:
stargazer(ols, ols,wls, type = "text",
          se= list(NULL, robust_se, NULL), column.labels = c("OLS", "OLS Robust s.e.", "WLS"))

Let's store this into a file now.

In [ ]:
stargazer(ols, ols,wls, type = "html", out ="ols&robust&wls.html",
          se= list(NULL, robust_se, NULL), column.labels = c("OLS", "OLS Robust s.e.", "WLS"))

# Robust F tests
Let's repeat some of our previous test but withour robust standard errors now.

In [ ]:
library("car")

Previously we had:

In [ ]:
linearHypothesis(ols, c("ecoprc=0","regprc=0"))

And now, letting R now that we want White's adjustment for robust standard errors:

In [ ]:
linearHypothesis(ols, c("ecoprc=0","regprc=0"), white.adjust = "hc1")

# FGLS
Now we will do some FGLS. Breusch-Pagan, with logs. Remember we assume that $V[u|x]=\sigma^2 e^{X\gamma}$ and therefore to do a linear regression, we take logs to "linearize" this expression.

In [ ]:
mydata$logolsresidualssq <- log(mydata$ols.residualssq)
bp = lm(logolsresidualssq~ecoprc+regprc+faminc+hhsize+educ+age, data = mydata)

mydata$fitbp <- exp(bp$fitted.values)
summary(mydata$fitbp)

Now we can create the Breush-Pagan weights

In [ ]:
mydata$weight_bp <- 1/(mydata$fitbp)

And run our model using these weights now. Let's immediately compare it against our other results using stargazer.

In [ ]:
fgls = lm(ecobuy~ecoprc+regprc+faminc+hhsize+educ+age, weight=mydata$weight_bp, data = mydata)
stargazer(ols, ols,wls,fgls, type = "text",
          se= list(NULL, robust_se,NULL,NULL), column.labels = c("OLS", "OLS Robust s.e.", "WLS", "FGLS"),
          omit.stat = "f")

In case you want to store this in a file:

In [ ]:
stargazer(ols, ols,wls,fgls, type = "html", out ="ols&robust&wls&fgls.html",
          se= list(NULL, robust_se,NULL,NULL), column.labels = c("OLS", "OLS Robust s.e.", "WLS", "FGLS"),
          omit.stat = "f")